# 03 — Deploy and predict through the UI

[Guide](../README.md) · [00: Data and Connections](00_setup_and_data.ipynb) · [01: Create an experiment](01_create_experiment.ipynb) · [02: Review results](02_review_results.ipynb) · [03: Deploy and predict](03_deploy_and_predict.ipynb)

[Equivalent code notebook](../03_predictor.ipynb)

Use the selected predictor and artifact Connection recorded in [02 — Review results](02_review_results.ipynb). Create a Model Registry, grant your project access, and register the selected predictor before deploying its version through **AI hub → Models → Registry**. Score both transactions using the terminal curl example in step 6.

You need permission to create a Model Registry and manage its access. If these administrator controls are unavailable, ask your cluster administrator to complete the registry creation and project-access steps. The deployment project also needs a Connection that can read the predictor's artifact bucket. This UI route does not create code-route state files.

## 1. Create a Model Registry

1. Open **Settings → AI hub → Model registry settings** in the dashboard (called **Settings → Model registries** in some releases).
2. Choose **Create model registry** and enter a name such as `fraud-automl-registry`.
3. Complete the required registry configuration for your cluster, then create it. Use the storage/database settings supplied by your administrator where the form requires them.
4. Wait until the registry is available before continuing. Record its name below; the registry is separate from the project where the model will be deployed.

| Setting | Your value |
| --- | --- |
| Model Registry | `<YOUR_MODEL_REGISTRY>` |
| Deployment project namespace | `<YOUR_PROJECT>` |

<!-- SCREENSHOT UI-14A: replace screenshots/03_1_create_model_registry.png; keep the filename. -->

<a href="screenshots/03_1_create_model_registry.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_1_create_model_registry.png" alt="UI-14A: Create the Model Registry" width="1500" style="max-width: 100%; height: auto;">
</a>

## 2. Add the project to the Model Registry

1. Open the registry's **Manage permissions** action on the **Model registry settings** page.
2. Add the project created in notebook 00, using its namespace (`<YOUR_PROJECT>`). Save the project-access configuration.
3. Check that the intended users can access the registry and that the project is listed. Registry access and permission to deploy models in the project are separate requirements.

This step grants the project access to the registry. Keep the artifact Connection in the deployment project as well; registry permissions do not grant S3 access.

<!-- SCREENSHOT UI-14B: replace screenshots/03_2_registry_project_access.png; keep the filename. -->

<a href="screenshots/03_2_registry_project_access.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_2_registry_project_access.png" alt="UI-14B: Project access to the Model Registry" width="1500" style="max-width: 100%; height: auto;">
</a>

## 3. Add the selected model to the Model Registry

1. Open the **AutoML** tab and select the completed training run whose model you want to deploy. Find the selected model in its results table.
2. Open that model row's **three-dot menu (⋮)** and choose **Register model**.
3. In the registration dialog, select the Model Registry created in step 1 (`<YOUR_MODEL_REGISTRY>`).
4. Enter or review the model name, such as `fraud-automl`, and the initial version name `v1`. Review any prefilled model details against the selected training run, then confirm **Register model**.
5. Open **AI hub → Models → Registry**, select the same registry, and confirm that the model and version **v1** are listed before continuing to deployment.

Use the registration action from the selected AutoML training run to keep the registered model associated with that run's output. Keep the predictor location recorded in notebook 02 for verification.

<!-- SCREENSHOT UI-14C: replace screenshots/03_3_register_model.png; keep the filename. -->

<a href="screenshots/03_3_register_model.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_3_register_model.png" alt="UI-14C: Register the model from AutoML" width="1500" style="max-width: 100%; height: auto;">
</a>

## 4. Deploy version v1 from the registry

1. Open **AI hub → Models → Registry**, select your registry, and find the registered model.
2. Open the model's **Actions** menu (or its details page) and choose **Deploy v1**.
3. Select the deployment project (`<YOUR_PROJECT>`) and enter a deployment name such as `fraud-automl` where prompted. Record the model's resource name used by the inference API; it can differ from its display name.
4. Set **Model type** to **Predictive model**.
5. Select the appropriate **Hardware profile**. For the environment used in this walkthrough, this is **`default-profile`**. Use a profile available in your cluster with enough CPU and memory for the predictor; `default-profile` is an example profile name, not a universal requirement.
6. Set **Model framework** to **`autogluon-1`**. This must use the compatible AutoGluon serving runtime (`kserve-autogluon-server`) for the managed training artifacts.
7. Verify the registered version's artifact location and choose the artifact-bucket Connection if requested. It must resolve to the complete selected `predictor/` directory.
8. Enable **Model access** and select **Require token authentication**. Keep authentication enabled for the prediction steps below.
9. Review the settings and click **Deploy model**.

If this version is already deployed, inspect that deployment before creating another one. Registering a new version does not automatically replace the predictor in an existing deployment.

<!-- SCREENSHOT UI-14: replace screenshots/03_4_deploy_model.png; keep the filename. -->

<a href="screenshots/03_4_deploy_model.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_4_deploy_model.png" alt="UI-14: Deploy registered AutoGluon model version v1" width="1500" style="max-width: 100%; height: auto;">
</a>

## 5. Wait for Ready and copy the endpoint

1. Open the deployment in the target project's **Models** page and wait until its status is **Ready**. Inspect events if it does not become ready.
2. Copy the **base inference endpoint**, including `https://` and without a `/v1/...` suffix.
3. Record the model resource name used by the inference API. The registry version name `v1` is not the model resource name in the request URL.
4. Because **Require token authentication** is enabled, obtain a current inference token using the deployment's supported model-access controls. Ensure that the token's identity has permission to call this model.
5. Keep the token only in the local terminal session for the curl request. Do not save it in this notebook, screenshots, or an exported shared collection.

A Ready status confirms deployment readiness, not a successful model prediction. For storage initialization errors, check the registered artifact location and deployment Connection/bucket/path. For prediction errors, inspect the runtime logs.

<!-- SCREENSHOT UI-15: replace screenshots/03_5_model_ready.png; keep the filename. -->

<a href="screenshots/03_5_model_ready.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_5_model_ready.png" alt="UI-15: Ready model and inference endpoint" width="1500" style="max-width: 100%; height: auto;">
</a>

## 6. Send both transactions with curl

Copy the three values below from the deployed model's UI and paste them into the variable assignments. Run all commands in the **same Bash or Zsh terminal** on a machine that can reach the cluster.

### A. Copy the inference endpoint

Open your project's **Models** page, select the Ready deployment, and copy its external **Inference endpoint**. Use the base HTTPS URL, without `/v1/models/...`.

```bash
PREDICT_ENDPOINT='https://<YOUR_INFERENCE_HOST>'
```

<!-- SCREENSHOT UI-16A: replace screenshots/03_6_inference_endpoint.png; keep the filename. -->

<a href="screenshots/03_6_inference_endpoint.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_6_inference_endpoint.png" alt="UI-16A: Where to copy PREDICT_ENDPOINT" width="1500" style="max-width: 100%; height: auto;">
</a>

### B. Copy the model resource name

In the same deployment's details, find the **model resource name** used by the inference API. If the UI offers a sample inference request, copy the name between `/v1/models/` and `:predict`. If it exposes the underlying **InferenceService** resource details/YAML, use `metadata.name`.

Use the deployed resource name, which can differ from the registered model's display name or version. Do not use the project name or just `v1`.

```bash
MODEL_NAME='<YOUR_MODEL_RESOURCE_NAME>'
```

<!-- SCREENSHOT UI-16B: replace screenshots/03_7_model_resource_name.png; keep the filename. -->

<a href="screenshots/03_7_model_resource_name.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_7_model_resource_name.png" alt="UI-16B: Where to copy MODEL_NAME" width="1500" style="max-width: 100%; height: auto;">
</a>

### C. Copy the inference token

On **AI hub → Models → Deployments** (or your project's **Models** page), expand the deployed model row to open its details. Under **Auth tokens**, use the copy control for a token authorized to call this model. If your release uses a separate **Model access** section, use its token controls instead. **Require token authentication** was enabled during deployment, so the request needs this token.

Paste **only the token** between the quotes below, without the `Bearer` prefix. This is a normal shell-variable assignment; there is no hidden prompt or clipboard command. The value is visible in the terminal and may be saved in shell history. Keep the real token out of the notebook, screenshots, and shared files.

```bash
PREDICT_TOKEN='<YOUR_INFERENCE_TOKEN>'
```

The token remains available in this terminal session for subsequent requests. Replace the value when it expires.

<!-- SCREENSHOT UI-16C: replace screenshots/03_8_inference_token.png; keep the filename. -->

<a href="screenshots/03_8_inference_token.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_8_inference_token.png" alt="UI-16C: Where to obtain PREDICT_TOKEN" width="1500" style="max-width: 100%; height: auto;">
</a>

### D. Send both transactions

After setting all three variables, run this request. It prints the response body followed by the HTTP status.

```bash
curl --silent --show-error --fail-with-body --request POST \
  "${PREDICT_ENDPOINT%/}/v1/models/${MODEL_NAME}:predict" \
  --header "Authorization: Bearer ${PREDICT_TOKEN}" \
  --header 'Content-Type: application/json' \
  --write-out '\nHTTP status: %{http_code}\n' \
  --data '{
  "instances": [
    {
      "distance_from_home": [0.5],
      "distance_from_last_transaction": [0.0],
      "ratio_to_median_purchase_price": [1.0],
      "repeat_retailer": [1.0],
      "used_chip": [1.0],
      "used_pin_number": [1.0],
      "online_order": [0.0]
    },
    {
      "distance_from_home": [150.0],
      "distance_from_last_transaction": [100.0],
      "ratio_to_median_purchase_price": [1.2],
      "repeat_retailer": [0.0],
      "used_chip": [0.0],
      "used_pin_number": [0.0],
      "online_order": [1.0]
    }
  ]
}'
```

Keep TLS verification enabled; for a private cluster CA, add `--cacert /path/to/cluster-ca.pem` to curl.

The one-element feature lists match the AutoGluon KServe v1 contract used by the code route. Instance 1 is Sally buying coffee locally; instance 2 is a suspicious online purchase. These are illustrative transactions; the model is not required to return a particular label for either one.

You can also send the same URL, Bearer token, and JSON body through Postman. If you prefer Python, use the code route's [predictor notebook](../03_predictor.ipynb) after completing its required run/results state; this UI route does not populate those files automatically.

## 7. Review the curl results

Inspect the terminal output from step 6. A successful smoke test requires **HTTP status: 200** and a JSON `predictions` array containing exactly two results, in request order:

1. Sally buys coffee locally.
2. Suspicious online purchase.

Record the **actual** returned values and HTTP status in the worksheet in step 8. Compare the output with the screenshot below, then record the response from your own deployment. The screenshot illustrates a previous request; your returned values can differ.

The response can contain class labels, such as 0 and 1. Do not interpret those labels as fraud probabilities or apply the original ONNX probability threshold. Use the official generated predictor notebook for `predict_proba` if needed.

| Symptom | What to check |
| --- | --- |
| HTTP 401/403 | Current inference credentials, configured authentication, and model access permissions |
| HTTP 404 | Base endpoint, model resource name, and `/v1/models/<YOUR_MODEL_NAME>:predict` path |
| HTTP 500 | Response body and `kserve-container` logs; a Ready model can still fail to load a dependent predictor |
| Wrong number of predictions | Request/response contract and runtime behavior |

For the known FastAI/Ray export/runtime dependency error, see [serving troubleshooting](../README.md#serving-and-troubleshooting). Installing a dependency in a notebook does not change the serving image.

<!-- SCREENSHOT UI-17: replace screenshots/03_9_prediction_results.png; keep the filename. -->

<a href="screenshots/03_9_prediction_results.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/03_9_prediction_results.png" alt="UI-17: curl response for both transactions" width="500" style="max-width: 100%; height: auto;">
</a>

## 8. Record evidence and clean up

This UI route has a manual evidence worksheet, independent of the code route's generated report. Fill it from the actual experiment and request before copying it to the ticket.

| Evidence | Your observation |
| --- | --- |
| Branch / PR | `<BRANCH_URL>` / `<PR_URL>` |
| Project / run ID | `<YOUR_PROJECT>` / `<RUN_ID>` |
| Training S3 URI | `<TRAIN_URI>` |
| Dataset input / evaluation handling | `Training CSV only; automatic evaluation; no separate evaluation data supplied` |
| Training rows / fraud rate | `<OBSERVED_TRAINING_STATISTICS>` |
| Evaluation rows / fraud rate, if exposed | `<OBSERVED_VALUES_OR_NOT_EXPOSED>` |
| Preset / pipeline version / duration | `<PRESET>` / `<VERSION>` / `<DURATION>` |
| Success, failure, or OOM evidence | `<OBSERVED_STATE_AND_EVIDENCE>` |
| Ranking metric / evaluation score | `<METRIC>` / `<OBSERVED_SCORE>` |
| Selected predictor URI | `<PREDICTOR_URI>` |
| Model Registry / registered model / version | `<YOUR_MODEL_REGISTRY>` / `<REGISTERED_MODEL_NAME>` / `v1` |
| Model name / endpoint | `<MODEL_NAME>` / `<BASE_ENDPOINT>` |
| Predict worked / HTTP status / actual values | `<TRUE_OR_FALSE>` / `<STATUS>` / `<PREDICTIONS_OR_ERROR>` |

Stop/delete your model and cancel unwanted runs through the dashboard when finished. Remove only your own input/artifact objects when no longer needed, preserving shared Connections, storage, and registries. Stopping a deployment does not delete its registered model version; remove only your own registry entries when no longer needed. If you chose a code deployment instead, use that route's cleanup instructions for its extra resources.

To repeat with `balanced`, reuse the same input objects and metric, create a separately named experiment, and record its own run ID and duration.

[Previous: Review results](02_review_results.ipynb) · [Return to the guide](../README.md)